# Domain 5: Context Management & Reliability
## TS 5.6: Preserve Information Provenance and Handle Uncertainty in Multi-Source Synthesis
**Certification Focus**: CCAR-F Domain 5 (15% Exam Weight) | **Scenario Alignment**: Scenario 3 & 6

---

### 🎯 Key Architectural Principles
1. **The Preservation Contract**: When multiple credible sources provide conflicting facts (e.g. Q3 revenue reported as $1.0B vs $1.2B), Claude must **never average the values**.
2. **Explicit Provenance Attribution**: Each conflicting claim must retain its source name, publication date, and measurement methodology.
3. **Highlighting Uncertainty**: Disclose the conflict clearly so human decision-makers understand the variance rather than acting on a false composite consensus.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 5.6 Provenance & Uncertainty.")

### 📜 Step 2: Provenance Tracking & Synthesis Engine
We implement the metric claim container and synthesis engine that strictly preserves conflicting values.

In [ ]:
# Step 2: Provenance Synthesizer Logic
@dataclass
class SourceCitation:
    source_name: str
    published_date: str
    methodology: str


@dataclass
class FactClaim:
    metric: str
    value: str
    citation: SourceCitation


@dataclass
class ProvenanceSynthesisEntry:
    metric: str
    has_conflict: bool
    status: str  # 'RESOLVED_SINGLE_SOURCE' | 'PRESERVED_CONFLICT'
    claims: List[Dict[str, Any]]
    synthesized_guidance: str


class ProvenanceSynthesizer:
    def __init__(self) -> None:
        self.claims: List[FactClaim] = []

    def add_claim(self, metric: str, value: str, source_name: str, published_date: str, methodology: str) -> None:
        self.claims.append(FactClaim(
            metric=metric,
            value=value,
            citation=SourceCitation(source_name=source_name, published_date=published_date, methodology=methodology)
        ))

    def synthesize_metrics(self) -> List[ProvenanceSynthesisEntry]:
        by_metric = {}
        for c in self.claims:
            by_metric.setdefault(c.metric, []).append(c)

        entries = []
        for metric, claims_list in by_metric.items():
            distinct_values = {c.value for c in claims_list}
            has_conflict = (len(distinct_values) > 1)

            claims_data = [
                {
                    "value": c.value,
                    "source": c.citation.source_name,
                    "date": c.citation.published_date,
                    "methodology": c.citation.methodology,
                }
                for c in claims_list
            ]

            if has_conflict:
                guidance = (
                    f"Contradictory values detected for '{metric}'. "
                    "CRITICAL: Do NOT calculate an average or blend these metrics. "
                    "Both values and sources are preserved explicitly for stakeholder review."
                )
                status = "PRESERVED_CONFLICT"
            else:
                guidance = f"Unanimous consensus on '{metric}'."
                status = "RESOLVED_SINGLE_SOURCE"

            entries.append(ProvenanceSynthesisEntry(
                metric=metric,
                has_conflict=has_conflict,
                status=status,
                claims=claims_data,
                synthesized_guidance=guidance,
            ))

        return entries

print(" ProvenanceSynthesizer ready.")

### 🧪 Step 3: Interactive Simulation
We record conflicting revenue numbers from a preliminary press release and an audited 10-Q filing.

In [ ]:
# Step 3: Simulation
synth = ProvenanceSynthesizer()

# Claim 1: Preliminary Press Release
synth.add_claim("Q3 Revenue", "$1.0B", "Press_Release", "2026-10-01", "Preliminary Unaudited")

# Claim 2: Audited 10-Q Filing
synth.add_claim("Q3 Revenue", "$1.2B", "SEC_10Q_Filing", "2026-10-15", "Audited Financials")

entries = synth.synthesize_metrics()
print("Synthesized Entries Count:", len(entries))
entry = entries[0]
print("Metric:", entry.metric)
print("Conflict Detected:", entry.has_conflict)
print("Status:", entry.status)
print("Guidance:", entry.synthesized_guidance)
print("\nClaims with Provenance:")
for c in entry.claims:
    print(f"  - {c['source']} ({c['date']}): {c['value']} [{c['methodology']}]")

### 🔬 Step 4: Verification Assertions
We assert that:
1. Conflict is identified.
2. Status is `PRESERVED_CONFLICT`.
3. Guidance strictly prohibits averaging.
4. Both source citations are preserved.

In [ ]:
# Step 4: Verification Checks
assert len(entries) == 1
assert entry.has_conflict is True
assert entry.status == "PRESERVED_CONFLICT"
assert "Do NOT calculate an average" in entry.synthesized_guidance
assert len(entry.claims) == 2

sources = [c["source"] for c in entry.claims]
assert "Press_Release" in sources
assert "SEC_10Q_Filing" in sources

print(" All verification assertions passed! Provenance preservation verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Never calculate an average or aggregate conflicting figures when authoritative sources disagree.
- **Rule 2**: Preserve all claims with source name, publication date, and measurement methodology.
- **Rule 3**: Disclose uncertainty explicitly to human decision-makers rather than presenting a false consensus.